# Random Forest Baseline
Train a simple Random Forest classifier on extracted pose features to compare against the neural models.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load the static feature dataset that was generated by the extraction workflow.
DATA_CSV = 'data/extracted_static_data.csv'
df = pd.read_csv(DATA_CSV)
print('Loaded', df.shape, 'rows')


Loaded (1811, 85) rows


In [ ]:
# Identify the feature columns and ensure the label column exists.
feature_columns = [
    c
    for c in df.columns
    if c not in ['Frame', 'Label', 'Label_Simplified', 'Device', 'Starting_Frame']
]
if 'Label_Simplified' not in df.columns:
    df['Label_Simplified'] = df['Label']

# Drop rows with missing feature values and normalize the feature matrix.
df = df.dropna(subset=feature_columns + ['Label_Simplified']).reset_index(drop=True)
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(0).values
encoder = LabelEncoder()
y = encoder.fit_transform(df['Label_Simplified'])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# Train a simple Random Forest baseline classifier and evaluate its performance.
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=encoder.classes_))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

      ground       0.62      0.84      0.72        63
        kick       0.55      0.68      0.61        62
 not_engaged       0.00      0.00      0.00        12
       punch       0.55      0.37      0.44        30
    takedown       0.25      0.05      0.08        20

    accuracy                           0.57       187
   macro avg       0.40      0.39      0.37       187
weighted avg       0.51      0.57      0.52       187

[[53  5  0  3  2]
 [13 42  2  5  0]
 [ 2  8  0  1  1]
 [ 2 17  0 11  0]
 [15  4  0  0  1]]
